# Load libraries

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import gc
import cv2
from PIL import Image
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.preprocessing import normalize

from tqdm.notebook import tqdm

import math
import psutil

import lightgbm as lgb
import sklearn.tree as skt
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, classification_report, plot_confusion_matrix
from sklearn.metrics import average_precision_score
from sklearn.utils.class_weight import compute_sample_weight

# Custom functions

In [ ]:
# Function to store pickle object
def pickle_dump(path, saveobj):
    import pickle
    filehandler = open(path,"wb")
    pickle.dump(saveobj,filehandler)
    print("File pickled")
    filehandler.close()

In [ ]:
# Function to load pickle object
def pickle_load(path):
    import pickle
    file = open(path,'rb')
    loadobj = pickle.load(file)
    file.close()
    return loadobj

# Prepare dataset

In [ ]:
path = "/kaggle/input/siimcovid19trainedmodelsstudyclassification/DL_preds.csv"
train = pd.read_csv(path)

print(train.shape)
train.head()

In [ ]:
path = "/kaggle/input/siim-covid19-detection/train_study_level.csv"
train_study_df = pd.read_csv(path)

print(train_study_df.shape)
train_study_df.head()

In [ ]:
train_study_df['id'] = train_study_df['id'].apply(lambda x: x.split('_')[0])
train_study_df.rename(columns={'id':'StudyInstanceUID'}, inplace=True)

train = pd.merge(train, train_study_df, on="StudyInstanceUID", how="left")

print(train.shape)
train.head()

# KFold Training

In [ ]:
print(train.columns)
train.head(2)

In [ ]:
modelCols = [col for col in train.columns if 'pred' in col]

# Separate input and target
X = train[modelCols]
y = train['class']

In [ ]:
from OneVsRestLightGBMWithCustomizedLoss import *
from FocalLoss import FocalLoss #get the FocalLoss implementation from Halford's blog

# Instantiate Focal loss
loss = FocalLoss(alpha=0.75, gamma=2.0)

# Not using early stopping
clf = OneVsRestLightGBMWithCustomizedLoss(loss=loss)
clf.fit(X_train, y_train)

# Using early stopping
#fit_params = {'eval_set': [(X_test, y_test)]}
#clf.fit(X_train, y_train, **fit_params)

y_test_pred = clf.predict(X_test)
pred_accuracy_score = accuracy_score(y_test, y_test_pred)
pred_recall_score = recall_score(y_test, y_test_pred, average='macro')
print('prediction accuracy', pred_accuracy_score,' recall ', pred_recall_score)

cnf_matrix = confusion_matrix(y_test, y_test_pred, labels=labeles)
plot_confusion_matrix(cnf_matrix, classes=classes,normalize=True,  title='Confusion matrix')

In [ ]:
# import ovr_lgbm_customloss as lgbm

# # Modeling
# folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

# oof_preds = np.zeros(X.shape[0])
# oof_preds_probs = np.zeros((X.shape[0],4))


# for n_fold, (trn_idx, val_idx) in enumerate(folds.split(X, y)):
#     trn_x, trn_y = X.iloc[trn_idx], y.iloc[trn_idx]
#     val_x, val_y = X.iloc[val_idx], y.iloc[val_idx]
    
#     trn_wts = compute_sample_weight(class_weight='balanced', y=trn_y)
#     val_wts = compute_sample_weight(class_weight='balanced', y=val_y)
    
#     # Instantiate Focal loss
#     loss = lgbm.FocalLoss(alpha=0.75, gamma=2.0)
    
# #     params = {
# #                 objective:"multiclass",
# #                 n_estimators:10000,
# #                 learning_rate:0.005,
# #                 num_leaves:20,
# #                 colsample_bytree:.6,
# #                 subsample:.7,
# #                 max_depth:4,
# #                 reg_alpha:0.1,
# #                 reg_lambda:10,
# # #                 min_split_gain=.01,
# # #                 min_child_weight=2

# #     }
    
# #     fit_params = {
# # #             objective:"multiclass",
# #             n_estimators:10000,
# #             learning_rate:0.005,
# #             num_leaves:20,
# #             colsample_bytree:.6,
# #             subsample:.7,
# #             max_depth:4,
# #             reg_alpha:0.1,
# #             reg_lambda:10,
# # #         sample_weight = trn_wts,
# #             eval_set= [(trn_x, trn_y), (val_x, val_y)], 
# # #             eval_sample_weight = [trn_wts, val_wts],
# #             eval_metric='auc_mu', 
# #             verbose=250, early_stopping_rounds=150
# #     }
    
# #     clf = lgb.LGBMClassifier(**params)
#     clf = lgbm.OneVsRestLightGBMWithCustomizedLoss(loss=loss)
#     clf.fit(trn_x, trn_y)

    
    
# #     clf.fit(trn_x, trn_y, sample_weight = trn_wts,
# #             eval_set= [(trn_x, trn_y), (val_x, val_y)], eval_sample_weight = [trn_wts, val_wts],
# #             eval_metric='auc_mu', 
# #             verbose=250, early_stopping_rounds=150
# #            )
    
    
# #     oof_preds[val_idx] = clf.predict_proba(val_x, num_iteration=clf.best_iteration_)#[:, 1]
#     oof_preds[val_idx] = clf.predict(val_x)#[:, 1]
#     oof_preds_probs[val_idx] = clf.predict_proba(val_x)#[:, 1]
    
# #     print('Fold %2d AUC : %.6f' % (n_fold + 1, roc_auc_score(val_y, oof_preds[val_idx], multi_class='ovr')))
#     del clf, trn_x, trn_y, val_x, val_y
#     gc.collect()
    
# # print('Full AUC score %.6f' % roc_auc_score(y, oof_preds, multi_class='ovr'))  

In [ ]:
# y_true = np.zeros((X.shape[0],4))

# for ind, label in enumerate(y):
#     y_true[ind][label]=1

In [ ]:
print('LGBM train roc-auc: {}'.format(roc_auc_score(y, oof_preds_probs, multi_class='ovr')))

In [ ]:
# Calculate mAP score
mAPloss = average_precision_score(y_true, oof_preds_probs)
print(f"mAP score: {mAPloss}")

In [ ]:
cnf_matrix = confusion_matrix(y, oof_preds, labels=y.unique())
plot_confusion_matrix(cnf_matrix, classes=classes,normalize=True,  title='Confusion matrix')

In [ ]:
print('LGBM train roc-auc: {}'.format(roc_auc_score(y, oof_preds, multi_class='ovr')))
print('LGBM train accuracy: {}'.format(accuracy_score(y, oof_preds)))
print('LGBM train Confusion matrix: \\n {}'.format(confusion_matrix(y, oof_preds[:, 1])))
print('LGBM train Classification report: \\n {}'.format(classification_report(y, oof_preds)))

In [ ]:
clf = skt.DecisionTreeClassifier(min_samples_leaf=100)
clf.fit(X, y)

# Save models

In [ ]:
path = "./DTree100.pkl"
pickle_dump(path, clf)

In [ ]:
clf = pickle_load("./DTree100.pkl")

preds = clf.predict(X)

print('LGBM train accuracy: {}'.format(accuracy_score(y, preds)))
print('LGBM train Confusion matrix: \\n {}'.format(confusion_matrix(y, preds)))
print('LGBM train Classification report: \\n {}'.format(classification_report(y, preds)))